This notebook computes the eddy current-induced magnetic field (Bz) around an aluminum plate with a square cutout, scanned along a line near a coil driven at 200 Hz, using Maxwell 3D's Eddy Current solver.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time

In [ ]:
## Create Desktop, Project, and Design ##

# creating Desktop object
# write the aedt(ansys electronics desktop) version and whether you are using a student license
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# Option to automatically save at regular intervals
# disabled because the simulation is run on a script basis
DT.disable_autosave()

# solution type
sol_type = "EddyCurrent"

# creating maxwell3D design object
# write solution type and whether you are using a student license
# when a design object is first created, it comes with a new project, and both are given arbitrary names
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type, student_version=True)
# odesign object, needed to use Ansys-recorded script commands
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week1_ex4"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)

# Create directory if it doesn't exist yet (safe to re-run; won't error if already there)
os.makedirs(dir, exist_ok=True)

desi_name = "Week1_ex4"


In [ ]:
## Save project and apply design name ##

proj = M3D.oproject

# Save project with the target file name (directory created above)
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

# Rename design to match the target design name
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
## Draw Geometry ##

# draw Stock

origin = [0, 0, -1]
sizes = [294, 294, 19]
stock = M3D.modeler.create_box(origin=origin, sizes=sizes, name="Stock", material="aluminum")

origin = [18, 18, -1]
sizes = [108,108, 19]
hole = M3D.modeler.create_box(origin=origin, sizes=sizes, name="Hole", material="air")

stock.subtract(tool_list=hole, keep_originals=False)


# draw Coil

origin = [119, 25, 49]
sizes = [150, 150, 100]
coil_hole = M3D.modeler.create_box(origin=origin, sizes=sizes, name="Coil_Hole", material="air")

edge_tmp = coil_hole.edges
edge_z = []
for e in edge_tmp :     # find edges by comparing the z-coordinate of each edge's midpoint
    if (e.midpoint[2]>50)&(e.midpoint[2]<148) :
        edge_z.append(e)

for e in edge_z :
    e.fillet(radius="25mm", setback="0mm")

origin = [94, 0, 49]
sizes = [200, 200, 100]
coil = M3D.modeler.create_box(origin=origin, sizes=sizes, name="Coil", material="air")

edge_tmp = coil.edges
edge_z = []
for e in edge_tmp :     # find edges by comparing the z-coordinate of each edge's midpoint
    if (e.midpoint[2]>50)&(e.midpoint[2]<148) :
        edge_z.append(e)

for e in edge_z :
    e.fillet(radius="50mm", setback="0mm")

coil.subtract(tool_list=coil_hole, keep_originals=False)

M3D.assign_material(assignment=coil, material="copper")


In [ ]:
## Create Coordinate System ##

origin = [200, 100, 0]
offset_cs = M3D.modeler.create_coordinate_system(origin=origin, reference_cs='Global', name="Offset", mode='axis', 
                                                 view='iso', x_pointing=None, y_pointing=None, psi=0, theta=0, phi=0, u=None)

In [ ]:
## Assign Current ##

M3D.modeler.set_working_coordinate_system(name=offset_cs)   # switch working coordinate system

M3D.modeler.section(assignment=coil, plane="XZ", create_new=True, section_cross_object=False)
coil_section = M3D.modeler.sheet_objects[-1]    # most recently created sheet object

M3D.modeler.split(assignment=coil_section, plane="YZ", sides="PositiveOnly", tool=None, split_crossing_objs=False, delete_invalid_objs=True)

current1 = M3D.assign_current(assignment=coil_section, amplitude="2742A", phase='0deg', solid=False, swap_direction=True, name="Current1")


In [ ]:
## Create Region ##

region = M3D.modeler.create_region(pad_value=300, pad_type='Percentage Offset', name='Region')

In [ ]:
## Create Dummy ##

M3D.modeler.set_working_coordinate_system(name="Global")   # switch to Global coordinate system

origin = [-3, 68, 30]
sizes = [300, 8, 8]
dummy = M3D.modeler.create_box(origin=origin, sizes=sizes, name="dummy", material="vacuum")

In [ ]:
## Set Eddy Effect ##

M3D.eddy_effects_on(assignment="Stock", enable_eddy_effects=True, enable_displacement_current=False)


In [ ]:
## Create Mesh ##

M3D.mesh.assign_length_mesh(assignment=dummy, inside_selection=True, maximum_length=1, maximum_elements=2000, name="Length1")

# the assign_length_mesh method can't be called consecutively due to some bug
# so we work around it by calling the module directly
oModule = oDesign.GetModule("MeshSetup")
oModule.AssignLengthOp(
	[
		"NAME:Length2",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["Stock"],
		"RestrictElem:="	, True,
		"NumMaxElem:="		, "2000",
		"RestrictLength:="	, False,
		"MaxLength:="		, "1mm"
	])

In [ ]:
## Configure Analysis Setup ##

# setup object
my_setup = M3D.create_setup(name="Setup1", setup_type="EddyCurrent")

# check the Analysis setup properties for the current solution type
# stored as a dictionary in the setup object's props attribute
display(my_setup.props)

In [ ]:
# modify the desired fields from the props displayed above

my_setup.props['MaximumPasses'] = 10
my_setup.props['PercentError'] = 0.2
my_setup.props['Frequency'] = "200Hz"

display(my_setup.props)

In [ ]:
## Run Analysis ##

my_setup.analyze()

In [ ]:
## Post Processing ##

points = []
points.append([0, 72,34])
points.append([288, 72,34])
field_line = M3D.modeler.create_polyline(points=points, name="Field_Line", non_model=True)


# field calculator setup

oModule = oDesign.GetModule("FieldsReporter")

oModule.AddNamedExpression("Bz_real", "Fields", 
	[
		"NameOfExpression:="	, ["<Bx,By,Bz>"],
		"Operation:="		, ["ScalarZ"],
		"Operation:="		, ["Real"],
		"Operation:="		, ["Smooth"],
		"Scalar_Constant:="	, [10000],
		"Operation:="		, ["*"]
	])

oModule.AddNamedExpression("Bz_img", "Fields", 
	[
		"NameOfExpression:="	, ["<Bx,By,Bz>"],
		"Operation:="		, ["ScalarZ"],
		"Operation:="		, ["Imag"],
		"Operation:="		, ["Smooth"],
		"Scalar_Constant:="	, [10000],
		"Operation:="		, ["*"]
	])

In [ ]:
## Save Project ##

M3D.save_project()